LOAD CHURN DATA

What we are doing:
Load churn dataset safely.

Why:
Churn data is large and can crash Python if handled incorrectly.

In [ ]:
import pandas as pd

churn_chunks = pd.read_csv(
    "/content/autoinsurance_churn.csv",
    chunksize=50000
)

churn = next(churn_chunks)


In [ ]:
churn.head()
churn.shape
churn.columns


Index(['individual_id', 'address_id', 'curr_ann_amt', 'days_tenure',
       'cust_orig_date', 'age_in_years', 'date_of_birth', 'latitude',
       'longitude', 'city', 'state', 'county', 'income', 'has_children',
       'length_of_residence', 'marital_status', 'home_market_value',
       'home_owner', 'college_degree', 'good_credit', 'acct_suspd_date',
       'Churn'],
      dtype='object')

CREATE CHURN FLAG
What we are doing

Create a clean churn indicator.

Why

We need a clear target to answer:

“Who left vs who stayed?”

In [ ]:
churn["CHURN_FLAG"] = churn["Churn"].map(
    {1: "Churned", 0: "Active"}
)


In [ ]:
churn[["Churn", "CHURN_FLAG"]].head()


,Churn,CHURN_FLAG
0,0,Active
1,0,Active
2,0,Active
3,1,Churned
4,0,Active


TENURE (CHURN RISK)

What we are doing:
Convert tenure from days to years and bucket it.

Why:
Short-tenure customers churn more; long-tenure customers are stickier.

In [ ]:
churn["TENURE_YEARS"] = churn["days_tenure"] / 365

churn["TENURE_BAND"] = pd.cut(
    churn["TENURE_YEARS"],
    bins=[0, 1, 3, 5, 50],
    labels=["New", "Early", "Mid", "Loyal"]
)


In [ ]:
churn[["days_tenure", "TENURE_YEARS", "TENURE_BAND"]].head()


,days_tenure,TENURE_YEARS,TENURE_BAND
0,1454.0,3.983562,Mid
1,1795.0,4.917808,Mid
2,4818.0,13.200000,Loyal
3,130.0,0.356164,New
4,5896.0,16.153425,Loyal


AGE BAND (CHURN SENSITIVITY)

What we are doing:
Group customers by age.

Why:
Churn behavior changes with age:

Younger → price sensitive

Older → stability but high value

In [ ]:
churn["AGE_BAND"] = pd.cut(
    churn["age_in_years"],
    bins=[0, 25, 40, 60, 100],
    labels=["Young", "Mid", "Senior", "Elder"]
)


In [ ]:
churn[["age_in_years", "AGE_BAND"]].head()


,age_in_years,AGE_BAND
0,44,Senior
1,72,Elder
2,55,Senior
3,53,Senior
4,50,Senior


INCOME BAND (CHURN vs AFFORDABILITY)

What we are doing:
Group customers by income level.

Why:
Income strongly affects:

price sensitivity

likelihood to churn after premium increases

We want balanced segments, so we use quantiles.

In [ ]:
churn["INCOME_BAND"] = pd.qcut(
    churn["income"],
    3,
    labels=["Low", "Medium", "High"]
)


In [ ]:
churn[["income", "INCOME_BAND"]].head()


,income,INCOME_BAND
0,22500.0,Low
1,27500.0,Low
2,42500.0,Low
3,125000.0,High
4,87500.0,Medium


PREMIUM BAND (PRICE SENSITIVITY)

What we are doing:
Group customers by how much premium they pay.

Why:
- Premium size strongly influences:
- churn risk
- negotiation power
- retention offers

In [ ]:
churn["PREMIUM_BAND"] = pd.qcut(
    churn["curr_ann_amt"],
    3,
    labels=["Low", "Medium", "High"]
)


In [ ]:
churn[["curr_ann_amt", "PREMIUM_BAND"]].head()


,curr_ann_amt,PREMIUM_BAND
0,818.877997,Low
1,974.199182,Medium
2,967.375112,Medium
3,992.409561,Medium
4,784.633494,Low


CREDIT RISK (FINANCIAL STABILITY)

What we are doing:
Convert credit quality into a clean churn-risk indicator.

Why:
Customers with poor credit are:

more price-sensitive

more likely to churn

In [ ]:
churn["CREDIT_RISK"] = churn["good_credit"].map(
    {1: "Good_Credit", 0: "Poor_Credit"}
)


In [ ]:
churn[["good_credit", "CREDIT_RISK"]].head()


,good_credit,CREDIT_RISK
0,1.0,Good_Credit
1,0.0,Poor_Credit
2,0.0,Poor_Credit
3,1.0,Good_Credit
4,1.0,Good_Credit


FAMILY STATUS (RETENTION BEHAVIOUR)

What we are doing:
Convert has_children into a clear family flag.

Why:
- Customers with families usually:
- stay longer
- are less price-sensitive
- respond better to retention offers

In [ ]:
churn["FAMILY_STATUS"] = churn["has_children"].map(
    {1: "Has_Children", 0: "No_Children"}
)


In [ ]:
churn[["has_children", "FAMILY_STATUS"]].head()


,has_children,FAMILY_STATUS
0,1.0,Has_Children
1,0.0,No_Children
2,0.0,No_Children
3,1.0,Has_Children
4,1.0,Has_Children


HOME OWNERSHIP (STABILITY SIGNAL)

What we are doing:
Create a home-ownership flag.

Why:
- Home owners tend to:
- stay longer
- churn less
- respond better to long-term pricing

In [ ]:
churn["HOME_STATUS"] = churn["home_owner"].map(
    {1: "Home_Owner", 0: "Renter"}
)


In [ ]:
churn[["home_owner", "HOME_STATUS"]].head()


,home_owner,HOME_STATUS
0,1.0,Home_Owner
1,1.0,Home_Owner
2,1.0,Home_Owner
3,1.0,Home_Owner
4,1.0,Home_Owner


In [ ]:
churn["HOME_STATUS"].value_counts()


,count
HOME_STATUS,
Home_Owner,40794
Renter,9206


SAVE PHASE 4 OUTPUT (CHURN FEATURES)

What we are doing:
Save the churn feature table we just built.

Why:
This file will be used for:

- churn analysis
- retention strategy
- modelling (Phase 5)


In [ ]:
churn.to_csv(
    "/content/churn_features_phase4.csv",
    index=False
)
